# Traditional NLP — Tutorial Notebook

** Natural Language Processing**  
Tutorial by Tamta Kapanadze · Supervised by Prof. Gözde Gül Şahin

---

## Contents

| Section | Topic |
|---------|-------|
| **1** | Text Representation Basics — Bag of Words & N-grams |
| **2** | N-gram Language Models — Bigram probabilities |
| **2S** | Smoothing & Discounting — Add-1, Add-k, Backoff, Interpolation |
| **3** | Perplexity — Evaluating language models |

**Corpus:** *The Little Prince*, Antoine de Saint-Exupéry (1943) — public domain  
**Reference:** Jurafsky & Martin, *Speech and Language Processing*, Ch. 3

---
# Section 1: Text Representation Basics

Bag of Words · N-grams

---
## 1.1 — Bag of Words (BoW)

The idea is simple: represent each sentence as a **vector of word counts**.  
We ignore word order — we just count how many times each word appears.

**Steps:**
1. Tokenize (split into words)
2. Build a shared vocabulary
3. Count occurrences of each word per sentence

In [2]:
# Our two sentences
s1 = "the food was not good the service was"
s2 = "the food was good the service was not"

# Step 1: Tokenize (split into words)
tokens_s1 = s1.split()
tokens_s2 = s2.split()

print("S1 tokens:", tokens_s1)
print("S2 tokens:", tokens_s2)

S1 tokens: ['the', 'food', 'was', 'not', 'good', 'the', 'service', 'was']
S2 tokens: ['the', 'food', 'was', 'good', 'the', 'service', 'was', 'not']


In [3]:
# Step 2: Build a shared vocabulary (sorted list of unique words)
vocab = sorted(set(tokens_s1 + tokens_s2))
print("Vocabulary:", vocab)

Vocabulary: ['food', 'good', 'not', 'service', 'the', 'was']


In [4]:
# Step 3: Count occurrences of each word in each sentence
def build_bow(tokens, vocab):
    return [tokens.count(word) for word in vocab]

bow_s1 = build_bow(tokens_s1, vocab)
bow_s2 = build_bow(tokens_s2, vocab)

print(f"{'Word':<12} {'S1':>4} {'S2':>4}")
print("-" * 22)
for word, c1, c2 in zip(vocab, bow_s1, bow_s2):
    print(f"{word:<12} {c1:>4} {c2:>4}")

Word           S1   S2
----------------------
food            1    1
good            1    1
not             1    1
service         1    1
the             2    2
was             2    2


In [5]:
# Are the two BoW vectors identical?
print("Are BoW vectors identical?", bow_s1 == bow_s2)

Are BoW vectors identical? True


### What does this mean?

S1 and S2 produce **identical BoW vectors** — even though one is negative and the other is (arguably) mixed.  
A classifier trained on BoW features would give **the same prediction** for both sentences.

> ⚠️ **Word order is completely lost in BoW.**

This is the core limitation we want to address.

---
## 1.2 — N-grams

**Idea:** Instead of individual words, look at sequences of N consecutive words.

| N | Name | Example from S1 |
|---|------|-----------------|
| 1 | Unigram | `"food"`, `"not"`, `"good"` |
| 2 | Bigram | `"was not"`, `"not good"` |
| 3 | Trigram | `"food was not"`, `"was not good"` |

N-grams capture **local word order** — they can tell `"was not good"` apart from `"was good"`.

In [6]:
# Extract n-grams from a list of tokens
def get_ngrams(tokens, n):
    #       join tokens[i : i+n] with a space
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

# Test on S1
print("--- S1 ---")
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(f"{name}: {get_ngrams(tokens_s1, n)}")

--- S1 ---
Unigrams: ['the', 'food', 'was', 'not', 'good', 'the', 'service', 'was']
Bigrams: ['the food', 'food was', 'was not', 'not good', 'good the', 'the service', 'service was']
Trigrams: ['the food was', 'food was not', 'was not good', 'not good the', 'good the service', 'the service was']


In [7]:
# Now compare bigrams of S1 and S2 side by side
bigrams_s1 = get_ngrams(tokens_s1, 2)
bigrams_s2 = get_ngrams(tokens_s2, 2)

print("Bigrams S1:", bigrams_s1)
print("Bigrams S2:", bigrams_s2)

# Which bigrams are unique to each sentence?
only_in_s1 = set(bigrams_s1) - set(bigrams_s2)
only_in_s2 = set(bigrams_s2) - set(bigrams_s1)

print("\nOnly in S1:", only_in_s1)
print("Only in S2:", only_in_s2)

Bigrams S1: ['the food', 'food was', 'was not', 'not good', 'good the', 'the service', 'service was']
Bigrams S2: ['the food', 'food was', 'was good', 'good the', 'the service', 'service was', 'was not']

Only in S1: {'not good'}
Only in S2: {'was good'}


### Key observation

Even with bigrams, we can already see the difference:
- S1 contains `"not good"` → negative signal
- S2 contains `"was good"` → positive signal

Bigrams **partially** recover word order. But they still can't capture long-range dependencies like:

> *"The computer which I had just put into the machine room on the fifth floor **crashed**"*

The subject (*computer*) and its verb (*crashed*) are far apart — a bigram model would miss this.

---
## 1.3 — Mini Exercise

Try it yourself! Given the sentence below, extract its **bigrams** and **trigrams**.

> *"I love natural language processing"*

In [8]:
exercise = "I love natural language processing"
tokens_ex = exercise.lower().split()

bigrams_ex = get_ngrams(tokens_ex, 2)
print("Bigrams:", bigrams_ex)

trigrams_ex = get_ngrams(tokens_ex, 3)
print("Trigrams:", trigrams_ex)

Bigrams: ['i love', 'love natural', 'natural language', 'language processing']
Trigrams: ['i love natural', 'love natural language', 'natural language processing']


---
## Section 1 Summary

| Method | Captures order? | Differentiates S1 vs S2? |
|--------|----------------|-------------------------|
| Bag of Words | ❌ | ❌ |
| Bigrams | Locally ✅ | ✅ |
| Trigrams | Better ✅ | ✅ |

➡️ **Next:** N-gram Language Models — we'll assign *probabilities* to word sequences and learn how to calculate P("I love NLP").

---
# Section 2: N-gram Language Models

Bigram probabilities · Chain rule · Zero probability problem

---
## 2.1 — Setup: Tokenize the Corpus

In [9]:
from collections import Counter

# Our corpus — The Little Prince (public domain)
corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)

# Tokenize
tokens = corpus.split()

print(f"Total tokens : {len(tokens)}")
print(f"Unique words : {len(set(tokens))}")
print(f"Tokens       : {tokens}")

Total tokens : 42
Unique words : 28
Tokens       : ['the', 'little', 'prince', 'went', 'away', 'he', 'went', 'to', 'look', 'at', 'the', 'roses', 'he', 'said', 'to', 'the', 'roses', 'you', 'are', 'not', 'at', 'all', 'like', 'my', 'rose', 'no', 'one', 'has', 'tamed', 'you', 'and', 'you', 'have', 'tamed', 'no', 'one', 'my', 'fox', 'was', 'like', 'that', 'once']


---
## 2.2 — Count Unigrams

A **unigram count** tells us how many times each word appears in the corpus.  
We need this to calculate bigram probabilities later.

In [10]:
# Count unigrams
unigram_counts = Counter(tokens)

print("Most common words:")
for word, count in unigram_counts.most_common(10):
    print(f"  {word:<12} {count}")

Most common words:
  the          3
  you          3
  went         2
  he           2
  to           2
  at           2
  roses        2
  like         2
  my           2
  no           2


In [11]:
print("Count of 'he'     :", unigram_counts["he"])
print("Count of 'the'    :", unigram_counts["the"])
print("Count of 'little' :", unigram_counts["little"])

Count of 'he'     : 2
Count of 'the'    : 3
Count of 'little' : 1


---
## 2.3 — Count Bigrams

A **bigram** is a pair of consecutive words.  
We count how many times each pair appears in the corpus.

In [12]:
# Extract all bigrams from the corpus
def get_bigrams(tokens):
    return list(zip(tokens, tokens[1:]))

bigrams = get_bigrams(tokens)
bigram_counts = Counter(bigrams)

print("Most common bigrams:")
for bigram, count in bigram_counts.most_common(8):
    print(f"  {str(bigram):<30} {count}")

Most common bigrams:
  ('the', 'roses')               2
  ('no', 'one')                  2
  ('the', 'little')              1
  ('little', 'prince')           1
  ('prince', 'went')             1
  ('went', 'away')               1
  ('away', 'he')                 1
  ('he', 'went')                 1


In [13]:
print("Count of ('he', 'said')    :", bigram_counts[("he", "said")])
print("Count of ('the', 'little') :", bigram_counts[("the", "little")])

Count of ('he', 'said')    : 1
Count of ('the', 'little') : 1


---
## 2.4 — From Counts to Probabilities

The bigram probability formula is:

$$P(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i)}{\text{Count}(w_{i-1})}$$

We divide the bigram count by the unigram count of the first word.

In [14]:
# Bigram probability function
def bigram_prob(w1, w2, bigram_counts, unigram_counts):
    if unigram_counts[w1] == 0:
        return 0.0
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

# Example: P(went | he)
p = bigram_prob("he", "went", bigram_counts, unigram_counts)
print(f"P(went | he)  = {bigram_counts[('he','went')]} / {unigram_counts['he']} = {p:.4f}")

P(went | he)  = 1 / 2 = 0.5000


In [15]:
p1 = bigram_prob("he", "said", bigram_counts, unigram_counts)
p2 = bigram_prob("the", "little", bigram_counts, unigram_counts)

print(f"P(said   | he)  = {p1:.4f}")
print(f"P(little | the) = {p2:.4f}")

P(said   | he)  = 0.5000
P(little | the) = 0.3333


---
## 2.5 — Calculate Sentence Probability

Using the **chain rule** + **bigram assumption**, the probability of a sentence is:

$$P(w_1 w_2 \ldots w_n) \approx P(w_1) \times \prod_{i=2}^{n} P(w_i \mid w_{i-1})$$

Let's calculate **P("he went to look")** step by step.

In [16]:
# Sentence probability function
def sentence_prob(sentence, unigram_counts, bigram_counts, total_tokens):
    words = sentence.lower().split()
    # Start with unigram probability of first word
    prob = unigram_counts[words[0]] / total_tokens
    print(f"  P({words[0]}) = {unigram_counts[words[0]]} / {total_tokens} = {prob:.4f}")
    # Multiply by each bigram probability
    for i in range(1, len(words)):
        w1, w2 = words[i-1], words[i]
        p = bigram_prob(w1, w2, bigram_counts, unigram_counts)
        print(f"  P({w2} | {w1}) = {bigram_counts[(w1,w2)]} / {unigram_counts[w1]} = {p:.4f}")
        prob *= p
    return prob

total_tokens = len(tokens)

print("Calculating P('he went to look'):")
prob = sentence_prob("he went to look", unigram_counts, bigram_counts, total_tokens)
print(f"\nFinal probability = {prob:.6f}")

Calculating P('he went to look'):
  P(he) = 2 / 42 = 0.0476
  P(went | he) = 1 / 2 = 0.5000
  P(to | went) = 1 / 2 = 0.5000
  P(look | to) = 1 / 2 = 0.5000

Final probability = 0.005952


---
## 2.6 — The Zero Probability Problem

What happens when a bigram has never been seen in our corpus?

In [17]:
# Try a sentence with an unseen bigram
print("Calculating P('the little prince said'):")
prob = sentence_prob("the little prince said", unigram_counts, bigram_counts, total_tokens)
print(f"\nFinal probability = {prob:.6f}")

print()
print("⚠️  'said' never followed 'prince' in our corpus.")
print("    P(said | prince) = 0  →  the entire sentence probability = 0")
print("    This is the Zero Probability Problem.")


Calculating P('the little prince said'):
  P(the) = 3 / 42 = 0.0714
  P(little | the) = 1 / 3 = 0.3333
  P(prince | little) = 1 / 1 = 1.0000
  P(said | prince) = 0 / 1 = 0.0000

Final probability = 0.000000

⚠️  'said' never followed 'prince' in our corpus.
    P(said | prince) = 0  →  the entire sentence probability = 0
    This is the Zero Probability Problem.


---
## 2.7 — Mini Exercise

Now it's your turn!  
Calculate **P("he said to the")** using the functions we built.

Expected steps:
- P(he)
- P(said | he)
- P(to | said)
- P(the | to)

In [18]:
print("Calculating P('he said to the'):")
prob = sentence_prob("he said to the", unigram_counts, bigram_counts, total_tokens)
print(f"\nFinal probability = {prob:.6f}")

Calculating P('he said to the'):
  P(he) = 2 / 42 = 0.0476
  P(said | he) = 1 / 2 = 0.5000
  P(to | said) = 1 / 1 = 1.0000
  P(the | to) = 1 / 2 = 0.5000

Final probability = 0.011905


---
## Section 2 Summary

| Concept | What we did |
|---------|-------------|
| Unigram counts | Counted how often each word appears |
| Bigram counts | Counted consecutive word pairs |
| Bigram probability | P(wᵢ \| wᵢ₋₁) = Count(wᵢ₋₁, wᵢ) / Count(wᵢ₋₁) |
| Sentence probability | Multiply unigram × bigram probabilities |
| Zero probability | Unseen bigrams → P = 0 → need smoothing |

➡️ **Next:** Perplexity — how do we measure how good a language model is?

---
# Section 2S: Smoothing & Discounting

Add-1 · Add-k · Stupid Backoff · Interpolation

---
## Setup: Corpus & Counts

In [19]:
from collections import Counter

# Our corpus — The Little Prince (public domain)
corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)

tokens = corpus.split()
unigram_counts = Counter(tokens)
bigram_counts  = Counter(zip(tokens, tokens[1:]))
total_tokens   = len(tokens)
V              = len(unigram_counts)  # vocabulary size

print(f"Total tokens : {total_tokens}")
print(f"Vocabulary   : {V}")
print(f"Unique bigrams: {len(bigram_counts)}")

Total tokens : 42
Vocabulary   : 28
Unique bigrams: 39


---
## 2S.1 — Baseline: MLE (No Smoothing)

$$P_{MLE}(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i)}{\text{Count}(w_{i-1})}$$

In [20]:
def mle_prob(w1, w2):
    if unigram_counts[w1] == 0:
        return 0.0
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

# Seen bigram
print(f"P(went  | prince) = {mle_prob('prince', 'went'):.4f}")
print(f"P(said  | he)     = {mle_prob('he', 'said'):.4f}")

# Unseen bigram
print(f"P(said  | prince) = {mle_prob('prince', 'said'):.4f}  ← zero!")
print(f"P(was   | rose)   = {mle_prob('rose', 'was'):.4f}  ← zero!")

P(went  | prince) = 1.0000
P(said  | he)     = 0.5000
P(said  | prince) = 0.0000  ← zero!
P(was   | rose)   = 0.0000  ← zero!


In [21]:
# Sentence probability with MLE
def sentence_prob_mle(sentence):
    words = sentence.lower().split()
    prob  = unigram_counts[words[0]] / total_tokens
    for i in range(1, len(words)):
        prob *= mle_prob(words[i-1], words[i])
    return prob

print(f"P('he went to look')         = {sentence_prob_mle('he went to look'):.6f}")
print(f"P('the little prince said')  = {sentence_prob_mle('the little prince said'):.6f}  ← zero!")

P('he went to look')         = 0.005952
P('the little prince said')  = 0.000000  ← zero!


---
## 2S.2 — Add-1 (Laplace) Smoothing

Add 1 to every bigram count — so nothing is ever zero.

$$P_{Add-1}(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i) + 1}{\text{Count}(w_{i-1}) + V}$$

In [22]:
def add1_prob(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

# Compare MLE vs Add-1
test_bigrams = [
    ("prince", "went"),
    ("he",     "said"),
    ("the",    "roses"),
    ("prince", "said"),  # unseen
    ("rose",   "was"),   # unseen
]

print(f"{'Bigram':<20} {'MLE':>8} {'Add-1':>10}")
print("-" * 42)
for w1, w2 in test_bigrams:
    mle  = mle_prob(w1, w2)
    add1 = add1_prob(w1, w2)
    flag = " ← was zero!" if mle == 0 else ""
    print(f"({w1}, {w2}){'':>{18-len(w1)-len(w2)-4}} {mle:>8.4f} {add1:>10.4f}{flag}")

Bigram                    MLE      Add-1
------------------------------------------
(prince, went)       1.0000     0.0690
(he, said)           0.5000     0.0667
(the, roses)         0.6667     0.0968
(prince, said)       0.0000     0.0345 ← was zero!
(rose, was)          0.0000     0.0345 ← was zero!


In [23]:
def sentence_prob_add1(sentence):
    words = sentence.lower().split()
    prob  = unigram_counts[words[0]] / total_tokens
    for i in range(1, len(words)):
        prob *= add1_prob(words[i-1], words[i])
    return prob

print(f"P('he went to look')        MLE   = {sentence_prob_mle('he went to look'):.6f}")
print(f"P('he went to look')        Add-1 = {sentence_prob_add1('he went to look'):.6f}")
print()
print(f"P('the little prince said') MLE   = {sentence_prob_mle('the little prince said'):.6f}  ← zero")
print(f"P('the little prince said') Add-1 = {sentence_prob_add1('the little prince said'):.6f}  ← no longer zero!")

P('he went to look')        MLE   = 0.005952
P('he went to look')        Add-1 = 0.000014

P('the little prince said') MLE   = 0.000000  ← zero
P('the little prince said') Add-1 = 0.000011  ← no longer zero!


---
## 2S.3 — Add-k Smoothing

Instead of adding 1, add a smaller fraction **k** — less aggressive than Add-1.

$$P_{Add-k}(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i) + k}{\text{Count}(w_{i-1}) + kV}$$

Common values: k = 0.5, k = 0.1, k = 0.01

In [24]:
def addk_prob(w1, w2, k):
    return (bigram_counts[(w1, w2)] + k) / (unigram_counts[w1] + k * V)

# Compare different values of k
w1, w2 = "prince", "said"  # unseen bigram
print(f"Unseen bigram: ({w1}, {w2})")
print(f"  MLE    : {mle_prob(w1, w2):.6f}")
for k in [1.0, 0.5, 0.1, 0.01]:
    print(f"  Add-{k:<4}: {addk_prob(w1, w2, k):.6f}")

print()
w1, w2 = "he", "said"  # seen bigram
print(f"Seen bigram: ({w1}, {w2})")
print(f"  MLE    : {mle_prob(w1, w2):.6f}")
for k in [1.0, 0.5, 0.1, 0.01]:
    print(f"  Add-{k:<4}: {addk_prob(w1, w2, k):.6f}")

Unseen bigram: (prince, said)
  MLE    : 0.000000
  Add-1.0 : 0.034483
  Add-0.5 : 0.033333
  Add-0.1 : 0.026316
  Add-0.01: 0.007812

Seen bigram: (he, said)
  MLE    : 0.500000
  Add-1.0 : 0.066667
  Add-0.5 : 0.093750
  Add-0.1 : 0.229167
  Add-0.01: 0.442982


Notice: as k gets smaller, Add-k approaches MLE for seen bigrams, but still gives non-zero probability to unseen ones.

---
## 2S.4 — Stupid Backoff

Simple and effective for large corpora (Brants et al., 2007).  
No discounting — just fall back to a lower-order model if the bigram is unseen.

$$S(w_i \mid w_{i-1}) = \begin{cases} \frac{\text{Count}(w_{i-1},\ w_i)}{\text{Count}(w_{i-1})} & \text{if Count}(w_{i-1}, w_i) > 0 \\ 0.4 \cdot \frac{\text{Count}(w_i)}{N} & \text{otherwise} \end{cases}$$

> Note: This is a **score**, not a true probability (does not sum to 1).

In [25]:
def stupid_backoff(w1, w2, alpha=0.4):
    # If bigram is seen → use MLE bigram probability
    # If bigram is unseen → back off to alpha * unigram probability
    if bigram_counts[(w1, w2)] > 0:
        return bigram_counts[(w1, w2)] / unigram_counts[w1]
    else:
        return alpha * (unigram_counts[w2] / total_tokens)

# Compare MLE vs Stupid Backoff
print(f"{'Bigram':<22} {'MLE':>8} {'Backoff':>10}")
print("-" * 44)
for w1, w2 in test_bigrams:
    mle = mle_prob(w1, w2)
    sb  = stupid_backoff(w1, w2)
    flag = " ← backed off" if mle == 0 else ""
    print(f"({w1}, {w2}){'':>{20-len(w1)-len(w2)-4}} {mle:>8.4f} {sb:>10.4f}{flag}")

Bigram                      MLE    Backoff
--------------------------------------------
(prince, went)         1.0000     1.0000
(he, said)             0.5000     0.5000
(the, roses)           0.6667     0.6667
(prince, said)         0.0000     0.0095 ← backed off
(rose, was)            0.0000     0.0095 ← backed off


---
## 2S.5 — Simple Interpolation

Combine trigram, bigram, and unigram models with weights that sum to 1.

$$\hat{P}(w_i \mid w_{i-2}w_{i-1}) = \lambda_1 P(w_i \mid w_{i-2}w_{i-1}) + \lambda_2 P(w_i \mid w_{i-1}) + \lambda_3 P(w_i)$$

where $\lambda_1 + \lambda_2 + \lambda_3 = 1$

Since our corpus is small we'll use **bigram + unigram interpolation** (λ₁ + λ₂ = 1).

In [26]:
def interpolated_prob(w1, w2, lambda1=0.7, lambda2=0.3):
    # Make sure lambda1 + lambda2 == 1
    assert abs(lambda1 + lambda2 - 1.0) < 1e-9, "Lambdas must sum to 1"
    bigram_p  = mle_prob(w1, w2)
    unigram_p = unigram_counts[w2] / total_tokens
    return lambda1 * bigram_p + lambda2 * unigram_p

# Compare all methods
print(f"{'Bigram':<22} {'MLE':>8} {'Add-1':>8} {'Backoff':>9} {'Interp':>9}")
print("-" * 60)
for w1, w2 in test_bigrams:
    mle   = mle_prob(w1, w2)
    add1  = add1_prob(w1, w2)
    sb    = stupid_backoff(w1, w2)
    interp = interpolated_prob(w1, w2)
    print(f"({w1}, {w2}){'':>{20-len(w1)-len(w2)-4}} {mle:>8.4f} {add1:>8.4f} {sb:>9.4f} {interp:>9.4f}")

Bigram                      MLE    Add-1   Backoff    Interp
------------------------------------------------------------
(prince, went)         1.0000   0.0690    1.0000    0.7143
(he, said)             0.5000   0.0667    0.5000    0.3571
(the, roses)           0.6667   0.0968    0.6667    0.4810
(prince, said)         0.0000   0.0345    0.0095    0.0071
(rose, was)            0.0000   0.0345    0.0095    0.0071


---
## 2S.6 — Full Comparison: Sentence Probabilities

Let's compare all methods on two sentences — one seen, one with an unseen bigram.

In [27]:
def sentence_prob_generic(sentence, prob_fn):
    """Calculate sentence probability using any bigram probability function."""
    words = sentence.lower().split()
    prob  = unigram_counts[words[0]] / total_tokens
    for i in range(1, len(words)):
        prob *= prob_fn(words[i-1], words[i])
    return prob

sentences = [
    "he went to look",
    "the little prince said",
]

methods = [
    ("MLE",          mle_prob),
    ("Add-1",        add1_prob),
    ("Add-0.1",      lambda w1, w2: addk_prob(w1, w2, 0.1)),
    ("Backoff",      stupid_backoff),
    ("Interpolation",interpolated_prob),
]

for sentence in sentences:
    print(f"\nP('{sentence}')")
    print("-" * 45)
    for name, fn in methods:
        p = sentence_prob_generic(sentence, fn)
        flag = "  ← ZERO" if p == 0 else ""
        print(f"  {name:<16}: {p:.8f}{flag}")


P('he went to look')
---------------------------------------------
  MLE             : 0.00595238
  Add-1           : 0.00001411
  Add-0.1         : 0.00057311
  Backoff         : 0.00595238
  Interpolation   : 0.00225687

P('the little prince said')
---------------------------------------------
  MLE             : 0.00000000  ← ZERO
  Add-1           : 0.00001096
  Add-0.1         : 0.00010320
  Backoff         : 0.00022676
  Interpolation   : 0.00008676


---
## 2S.7 — Mini Exercise

Try calculating the smoothed probability of a new sentence: **"the fox was like that"**

This sentence has:
- `(fox, was)` — seen ✅
- `(was, like)` — seen ✅  
- `(like, that)` — seen ✅

So MLE should work fine here. Let's verify, then compare with smoothed methods.

In [28]:
sentence = "the fox was like that"

print(f"P('{sentence}')")
print("-" * 45)
for name, fn in methods:
    p = sentence_prob_generic(sentence, fn)
    print(f"  {name:<16}: {p:.8f}")

P('the fox was like that')
---------------------------------------------
  MLE             : 0.00000000
  Add-1           : 0.00000073
  Add-0.1         : 0.00002365
  Backoff         : 0.00034014
  Interpolation   : 0.00009204


---
## Summary

| Method | Zero prob? | Formula | Notes |
|--------|-----------|---------|-------|
| MLE | ❌ Yes | Count(w₁,w₂) / Count(w₁) | Baseline |
| Add-1 | ✅ No | (Count+1) / (Count+V) | Too aggressive |
| Add-k | ✅ No | (Count+k) / (Count+kV) | Tune k |
| Stupid Backoff | ✅ No | MLE or 0.4 × unigram | Not a true prob |
| Interpolation | ✅ No | λ₁×bigram + λ₂×unigram | Tune λ weights |

➡️ **Next:** Perplexity — how do we measure how good a language model actually is?

---
# Section 3: Perplexity

Evaluating language models · Log probabilities · Comparing methods

---
## Setup: Reusing our corpus and models from Sections 2 & 2S

In [29]:
import math
from collections import Counter

# Corpus — The Little Prince (public domain)
corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)

tokens         = corpus.split()
unigram_counts = Counter(tokens)
bigram_counts  = Counter(zip(tokens, tokens[1:]))
total_tokens   = len(tokens)
V              = len(unigram_counts)

# Probability functions from Section 2
def mle_prob(w1, w2):
    if unigram_counts[w1] == 0: return 0.0
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

def add1_prob(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

def addk_prob(w1, w2, k=0.1):
    return (bigram_counts[(w1, w2)] + k) / (unigram_counts[w1] + k * V)

def stupid_backoff(w1, w2, alpha=0.4):
    if bigram_counts[(w1, w2)] > 0:
        return bigram_counts[(w1, w2)] / unigram_counts[w1]
    return alpha * (unigram_counts[w2] / total_tokens)

def interpolated_prob(w1, w2, lambda1=0.7, lambda2=0.3):
    return lambda1 * mle_prob(w1, w2) + lambda2 * (unigram_counts[w2] / total_tokens)

print("✅ Setup complete")
print(f"   Corpus: {total_tokens} tokens, V={V}")

✅ Setup complete
   Corpus: 42 tokens, V=28


---
## 3.1 — Sentence Log Probability

In practice we use **log probabilities** to avoid numerical underflow  
(multiplying many small numbers together quickly becomes 0.0 in floating point).

$$\log P(w_1 \ldots w_n) = \log P(w_1) + \sum_{i=2}^{n} \log P(w_i \mid w_{i-1})$$

In [30]:
def sentence_log_prob(sentence, prob_fn):
    """Returns log probability of a sentence under a bigram model."""
    words  = sentence.lower().split()
    # Start with log unigram prob of first word
    log_p  = math.log(unigram_counts[words[0]] / total_tokens)
    for i in range(1, len(words)):
        p = prob_fn(words[i-1], words[i])
        if p == 0:
            return float('-inf')  # log(0) = -infinity
        log_p += math.log(p)
    return log_p

# Test on a seen sentence
s = "he went to look at the roses"
lp_mle  = sentence_log_prob(s, mle_prob)
lp_add1 = sentence_log_prob(s, add1_prob)

print(f"Sentence: '{s}'")
print(f"  MLE  log P = {lp_mle:.4f}   P = {math.exp(lp_mle):.8f}")
print(f"  Add1 log P = {lp_add1:.4f}   P = {math.exp(lp_add1):.8f}")

Sentence: 'he went to look at the roses'
  MLE  log P = -6.2226   P = 0.00198413
  Add1 log P = -18.8862   P = 0.00000001


In [31]:
# Test on a sentence with an unseen bigram
s2 = "the little prince said"
lp_mle2  = sentence_log_prob(s2, mle_prob)
lp_add12 = sentence_log_prob(s2, add1_prob)

print(f"Sentence: '{s2}'")
print(f"  MLE  log P = {lp_mle2}  ← -inf because of zero bigram!")
print(f"  Add1 log P = {lp_add12:.4f}   P = {math.exp(lp_add12):.10f}")

Sentence: 'the little prince said'
  MLE  log P = -inf  ← -inf because of zero bigram!
  Add1 log P = -11.4213   P = 0.0000109591


---
## 3.2 — Perplexity Formula

For a test set W of N words:

$$PP(W) = P(w_1 w_2 \ldots w_N)^{-1/N}$$

In log space:

$$\log PP(W) = -\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid w_{i-1})$$

$$PP(W) = e^{-\frac{1}{N} \sum \log P(w_i \mid w_{i-1})}$$

In [32]:
def perplexity(test_sentences, prob_fn):
    """
    Compute perplexity of a model on a list of test sentences.
    Returns inf if any sentence has zero probability under the model.
    """
    #   1. Sum up log probabilities of all sentences
    #   2. Count total number of words
    #   3. Return exp(-total_log / total_words)
    total_log   = 0.0
    total_words = 0
    for s in test_sentences:
        lp = sentence_log_prob(s, prob_fn)
        if lp == float('-inf'):
            return float('inf')
        total_log   += lp
        total_words += len(s.split())
    return math.exp(-total_log / total_words)

# Test on a single sentence
s = "he went to look at the roses"
print(f"Sentence: '{s}'")
print(f"  Words: {len(s.split())}")
print(f"  MLE  PPL = {perplexity([s], mle_prob):.4f}")
print(f"  Add1 PPL = {perplexity([s], add1_prob):.4f}")

Sentence: 'he went to look at the roses'
  Words: 7
  MLE  PPL = 2.4325
  Add1 PPL = 14.8505


---
## 3.3 — Digit Intuition

Before evaluating our LMs, let's verify the digit example from the slides.

In [33]:
# Uniform distribution: P(each digit) = 1/10
# Test set of 10 digits: any sequence
N = 10
log_p_uniform = N * math.log(1/10)   # each digit has prob 1/10
ppl_uniform   = math.exp(-log_p_uniform / N)
print(f"Uniform distribution (10 digits):")
print(f"  Log P = {log_p_uniform:.4f}")
print(f"  PPL   = {ppl_uniform:.1f}")

print()

# Skewed: P(0)=0.91, P(others)=0.01
# Test: 0 0 0 0 0 3 0 0 0 0
test_probs    = [0.91]*9 + [0.01]   # nine 0s and one 3
log_p_skewed  = sum(math.log(p) for p in test_probs)
ppl_skewed    = math.exp(-log_p_skewed / N)
print(f"Skewed distribution (P(0)=0.91, test: 0 0 0 0 0 3 0 0 0 0):")
print(f"  Log P = {log_p_skewed:.4f}")
print(f"  PPL   = {ppl_skewed:.3f}")
print()
print("Uniform PPL=10 means the model always has 10 equally likely choices.")
print("Skewed  PPL≈1.7 means the model is usually very confident (picks 0 most of the time).")

Uniform distribution (10 digits):
  Log P = -23.0259
  PPL   = 10.0

Skewed distribution (P(0)=0.91, test: 0 0 0 0 0 3 0 0 0 0):
  Log P = -5.4540
  PPL   = 1.725

Uniform PPL=10 means the model always has 10 equally likely choices.
Skewed  PPL≈1.7 means the model is usually very confident (picks 0 most of the time).


---
## 3.4 — Comparing Models on Test Sentences

Let's evaluate all our smoothing methods on the same test set.

In [34]:
# Test sentences — from our Little Prince corpus
test_sentences = [
    "he went to look at the roses",
    "the little prince went away",
    "he said to the roses you are not",
]

methods = [
    ("MLE",           mle_prob),
    ("Add-1",         add1_prob),
    ("Add-0.1",       lambda w1, w2: addk_prob(w1, w2, k=0.1)),
    ("Backoff",       stupid_backoff),
    ("Interpolation", interpolated_prob),
]

# ── Per sentence perplexity ──────────────────────────────────
print("Per sentence perplexity:")
print()
col_w = 36
header = f"{'Sentence':<{col_w}}" + "".join(f"{name:>14}" for name, _ in methods)
print(header)
print("-" * (col_w + 14 * len(methods)))
for s in test_sentences:
    row = f"'{s}'"[:col_w-1].ljust(col_w)
    for name, fn in methods:
        ppl = perplexity([s], fn)
        cell = "INF" if ppl == float('inf') else f"{ppl:.2f}"
        row += f"{cell:>14}"
    print(row)

# ── Overall perplexity ───────────────────────────────────────
print()
print("Overall perplexity (all sentences combined):")
print()
print(f"{'Method':<16} {'PPL':>10}")
print("-" * 30)
for name, fn in methods:
    ppl = perplexity(test_sentences, fn)
    flag = "  ← INFINITE (zero prob)" if ppl == float('inf') else ""
    print(f"{name:<16} {ppl:>10.4f}{flag}")

Per sentence perplexity:

Sentence                                       MLE         Add-1       Add-0.1       Backoff Interpolation
----------------------------------------------------------------------------------------------------------
'he went to look at the roses'                2.43         14.85          4.95          2.43          3.20
'the little prince went away'                 2.43         14.69          5.21          2.43          3.18
'he said to the roses you are not'            2.29         14.87          4.84          2.29          3.04

Overall perplexity (all sentences combined):

Method                  PPL
------------------------------
MLE                  2.3728
Add-1               14.8171
Add-0.1              4.9704
Backoff              2.3728
Interpolation        3.1306


### Why does MLE win on these sentences?

MLE gives the lowest perplexity here because **all three test sentences are from the training corpus**.
MLE assigns them very high probability — but would fail completely on any unseen sentence.

This is why we always evaluate on a **held-out test set** that was **not** used for training!

---
## 3.5 — Unseen Test Sentences

Now let's use sentences that were **not** in our training corpus.

In [35]:
# Unseen test sentences — same words but different combinations
unseen_sentences = [
    "the prince looked at the rose",
    "he looked at the fox",
    "the little fox went away",
]

print("Test set: UNSEEN sentences")
print()

# ── Per sentence ─────────────────────────────────────────────
print("Per sentence perplexity:")
print()
col_w = 32
header = f"{'Sentence':<{col_w}}" + "".join(f"{name:>14}" for name, _ in methods)
print(header)
print("-" * (col_w + 14 * len(methods)))
for s in unseen_sentences:
    row = f"'{s}'"[:col_w-1].ljust(col_w)
    for name, fn in methods:
        ppl = perplexity([s], fn)
        cell = "INF" if ppl == float('inf') else f"{ppl:.2f}"
        row += f"{cell:>14}"
    print(row)

# ── Overall ──────────────────────────────────────────────────
print()
print("Overall perplexity (all sentences combined):")
print()
print(f"{'Method':<16} {'PPL':>12}")
print("-" * 32)
for name, fn in methods:
    ppl = perplexity(unseen_sentences, fn)
    flag = "  ← INFINITE" if ppl == float('inf') else ""
    print(f"{name:<16} {ppl:>12.4f}{flag}")

Test set: UNSEEN sentences

Per sentence perplexity:

Sentence                                   MLE         Add-1       Add-0.1       Backoff Interpolation
------------------------------------------------------------------------------------------------------
'the prince looked at the rose'            INF         23.39         24.54           INF           INF
'he looked at the fox'                     INF         24.14         23.48           INF           INF
'the little fox went away'                 INF         19.39         13.60         13.59         17.41

Overall perplexity (all sentences combined):

Method                    PPL
--------------------------------
MLE                       inf  ← INFINITE
Add-1                 22.2773
Add-0.1               20.1293
Backoff                   inf  ← INFINITE
Interpolation             inf  ← INFINITE


Results breakdown:

- MLE fails on all sentences — any unseen bigram or OOV word gives P=0
- Backoff and Interpolation survive only when all words are in the vocabulary
  (e.g. "the little fox went away" — all words seen, just new combinations)
  but fail on OOV words like "looked" which never appeared in our corpus
- Add-1 and Add-k are the only methods that handle truly OOV words
  because they add counts for every word pair regardless of whether the word was seen

Key distinction:
  Unseen bigram (known words) → Backoff/Interpolation can help
  Unseen word (OOV)           → only Add-1 / Add-k survive

1.   List item
2.   List item



---
## 3.6 — Mini Exercise

Calculate the perplexity of each model on the sentence below.  
This sentence has one unseen bigram — can you find which one?

> *"the rose tamed the fox"*

In [36]:
exercise = "the rose tamed the fox"

# First, find the unseen bigram
words = exercise.split()
print("Bigrams in the sentence:")
for i in range(len(words)-1):
    w1, w2 = words[i], words[i+1]
    count = bigram_counts[(w1, w2)]
    flag  = "  ← UNSEEN" if count == 0 else f"  count={count}"
    print(f"  ({w1}, {w2}){flag}")

print()

print(f"{'Method':<16} {'PPL':>12}")
print("-" * 32)
for name, fn in methods:
    ppl = perplexity([exercise], fn)
    flag = "  ← INFINITE" if ppl == float('inf') else ""
    print(f"{name:<16} {ppl:>12.4f}{flag}")

Bigrams in the sentence:
  (the, rose)  ← UNSEEN
  (rose, tamed)  ← UNSEEN
  (tamed, the)  ← UNSEEN
  (the, fox)  ← UNSEEN

Method                    PPL
--------------------------------
MLE                       inf  ← INFINITE
Add-1                 25.9223
Add-0.1               38.6191
Backoff               49.0396
Interpolation         61.7302


---
## Section 3 Summary

| Concept | Key point |
|---------|----------|
| Perplexity | Inverse probability of test set, normalized by N |
| Lower is better | Model is less surprised = better fit |
| Log probabilities | Avoid numerical underflow in practice |
| MLE on training data | PPL looks great — but fails on unseen data |
| Smoothing + PPL | Higher PPL on seen data, but handles unseen gracefully |
| Same vocabulary | Only valid to compare models with same V |

**WSJ reference results (Jurafsky & Martin, Ch. 3):**

| Model | PPL |
|-------|-----|
| Unigram | 962 |
| Bigram | 170 |
| Trigram | 109 |

More context → lower perplexity → better model.